In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

raw_path = Path("../data/raw")
processed_path = Path("../data/processed")

processed_path.mkdir(exist_ok=True)

print("Environment Ready")

Environment Ready


In [3]:
fund_master = pd.read_csv(raw_path / "01_fund_master.csv")
nav_history = pd.read_csv(raw_path / "02_nav_history.csv")
aum = pd.read_csv(raw_path / "03_aum_by_fund_house.csv")
sip = pd.read_csv(raw_path / "04_monthly_sip_inflows.csv")
category = pd.read_csv(raw_path / "05_category_inflows.csv")
folio = pd.read_csv(raw_path / "06_industry_folio_count.csv")
performance = pd.read_csv(raw_path / "07_scheme_performance.csv")
transactions = pd.read_csv(raw_path / "08_investor_transactions.csv")
holdings = pd.read_csv(raw_path / "09_portfolio_holdings.csv")
benchmark = pd.read_csv(raw_path / "10_benchmark_indices.csv")

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [4]:
print("Fund Master:", fund_master.shape)
print("NAV History:", nav_history.shape)
print("AUM:", aum.shape)
print("SIP:", sip.shape)
print("Category:", category.shape)
print("Folio:", folio.shape)
print("Performance:", performance.shape)
print("Transactions:", transactions.shape)
print("Holdings:", holdings.shape)
print("Benchmark:", benchmark.shape)

Fund Master: (40, 15)
NAV History: (46000, 3)
AUM: (90, 5)
SIP: (48, 6)
Category: (144, 3)
Folio: (21, 6)
Performance: (40, 19)
Transactions: (32778, 13)
Holdings: (322, 8)
Benchmark: (8050, 3)


In [5]:
datasets = {
    "fund_master": fund_master,
    "nav_history": nav_history,
    "aum": aum,
    "sip": sip,
    "category": category,
    "folio": folio,
    "performance": performance,
    "transactions": transactions,
    "holdings": holdings,
    "benchmark": benchmark
}

for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

fund_master: 0 duplicate rows
nav_history: 0 duplicate rows
aum: 0 duplicate rows
sip: 0 duplicate rows
category: 0 duplicate rows
folio: 0 duplicate rows
performance: 0 duplicate rows
transactions: 0 duplicate rows
holdings: 0 duplicate rows
benchmark: 0 duplicate rows


In [6]:
datasets = {
    "fund_master": fund_master,
    "nav_history": nav_history,
    "aum": aum,
    "sip": sip,
    "category": category,
    "folio": folio,
    "performance": performance,
    "transactions": transactions,
    "holdings": holdings,
    "benchmark": benchmark
}

for name, df in datasets.items():
    print("="*60)
    print(name)
    print("Shape:", df.shape)
    print("Duplicates:", df.duplicated().sum())
    print("Missing Values:")
    print(df.isnull().sum().sum())

fund_master
Shape: (40, 15)
Duplicates: 0
Missing Values:
0
nav_history
Shape: (46000, 3)
Duplicates: 0
Missing Values:
0
aum
Shape: (90, 5)
Duplicates: 0
Missing Values:
0
sip
Shape: (48, 6)
Duplicates: 0
Missing Values:
12
category
Shape: (144, 3)
Duplicates: 0
Missing Values:
0
folio
Shape: (21, 6)
Duplicates: 0
Missing Values:
0
performance
Shape: (40, 19)
Duplicates: 0
Missing Values:
0
transactions
Shape: (32778, 13)
Duplicates: 0
Missing Values:
0
holdings
Shape: (322, 8)
Duplicates: 0
Missing Values:
0
benchmark
Shape: (8050, 3)
Duplicates: 0
Missing Values:
0


In [7]:
fund_master["launch_date"] = pd.to_datetime(
    fund_master["launch_date"]
)

nav_history["date"] = pd.to_datetime(
    nav_history["date"]
)

aum["date"] = pd.to_datetime(
    aum["date"]
)

transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

holdings["portfolio_date"] = pd.to_datetime(
    holdings["portfolio_date"]
)

benchmark["date"] = pd.to_datetime(
    benchmark["date"]
)

print("Date conversion complete")

Date conversion complete


In [8]:
sip.isnull().sum()

month                         0
sip_inflow_crore              0
active_sip_accounts_crore     0
new_sip_accounts_lakh         0
sip_aum_lakh_crore            0
yoy_growth_pct               12
dtype: int64

In [9]:
sip["yoy_growth_pct"] = sip["yoy_growth_pct"].fillna(0)

In [10]:
sip.isnull().sum()

month                        0
sip_inflow_crore             0
active_sip_accounts_crore    0
new_sip_accounts_lakh        0
sip_aum_lakh_crore           0
yoy_growth_pct               0
dtype: int64

In [11]:
fund_master.to_csv(
    processed_path / "fund_master_clean.csv",
    index=False
)

nav_history.to_csv(
    processed_path / "nav_history_clean.csv",
    index=False
)

performance.to_csv(
    processed_path / "scheme_performance_clean.csv",
    index=False
)

transactions.to_csv(
    processed_path / "transactions_clean.csv",
    index=False
)

print("Processed files saved")

Processed files saved


## NAV History Cleaning

In [12]:
# Create working copy

nav_clean = nav_history.copy()

# Sort by AMFI code and Date

nav_clean = nav_clean.sort_values(
    ["amfi_code", "date"]
).reset_index(drop=True)

print(nav_clean.head())

   amfi_code       date       nav
0     100016 2022-01-03  520.4608
1     100016 2022-01-04  515.0971
2     100016 2022-01-05  521.7239
3     100016 2022-01-06  515.7880
4     100016 2022-01-07  515.1639


In [13]:
invalid_nav = nav_clean[
    nav_clean["nav"] <= 0
]

print("Invalid NAV Records:", len(invalid_nav))

Invalid NAV Records: 0


In [14]:
print(nav_clean.isnull().sum())

amfi_code    0
date         0
nav          0
dtype: int64


In [15]:
sample_fund = nav_clean[
    nav_clean["amfi_code"] == nav_clean["amfi_code"].iloc[0]
]

print("Start Date:", sample_fund["date"].min())
print("End Date:", sample_fund["date"].max())
print("Trading Days:", len(sample_fund))

Start Date: 2022-01-03 00:00:00
End Date: 2026-05-29 00:00:00
Trading Days: 1150


In [16]:
# Create full daily calendar for each fund

nav_ffill = []

for code, group in nav_clean.groupby("amfi_code"):

    group = group.set_index("date")

    full_dates = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq="D"
    )

    group = group.reindex(full_dates)

    group["amfi_code"] = code

    group["nav"] = group["nav"].ffill()

    group = group.reset_index()

    group.rename(
        columns={"index": "date"},
        inplace=True
    )

    nav_ffill.append(group)

nav_clean = pd.concat(
    nav_ffill,
    ignore_index=True
)

print(nav_clean.shape)

(64320, 3)


In [17]:
print(nav_clean.isnull().sum())

date         0
amfi_code    0
nav          0
dtype: int64


In [18]:
nav_clean.to_csv(
    processed_path / "clean_nav.csv",
    index=False
)

print("clean_nav.csv saved")

clean_nav.csv saved


## Investor Transactions Cleaning

In [19]:
transactions_clean = transactions.copy()

print(transactions_clean.shape)

(32778, 13)


In [20]:
print(
    transactions_clean["transaction_type"]
    .value_counts()
)

transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64


In [21]:
invalid_amounts = transactions_clean[
    transactions_clean["amount_inr"] <= 0
]

print(
    "Invalid Amount Records:",
    len(invalid_amounts)
)

Invalid Amount Records: 0


In [22]:
print(
    transactions_clean["kyc_status"]
    .value_counts()
)

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


In [23]:
print(
    transactions_clean.isnull().sum()
)

investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64


In [24]:
transactions_clean.to_csv(
    processed_path / "clean_transactions.csv",
    index=False
)

print("clean_transactions.csv saved")

clean_transactions.csv saved


## Scheme Performance Cleaning

In [25]:
performance_clean = performance.copy()

print(performance_clean.shape)

(40, 19)


In [26]:
performance_clean.dtypes

amfi_code               int64
scheme_name               str
fund_house                str
category                  str
plan                      str
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade                str
dtype: object

In [27]:
return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct"
]

for col in return_cols:
    print(
        col,
        pd.to_numeric(
            performance_clean[col],
            errors="coerce"
        ).isnull().sum()
    )

return_1yr_pct 0
return_3yr_pct 0
return_5yr_pct 0


In [28]:
negative_sharpe = performance_clean[
    performance_clean["sharpe_ratio"] < 0
]

print(
    "Negative Sharpe Funds:",
    len(negative_sharpe)
)

Negative Sharpe Funds: 0


In [29]:
invalid_expense = performance_clean[
    (performance_clean["expense_ratio_pct"] < 0.1)
    |
    (performance_clean["expense_ratio_pct"] > 2.5)
]

print(
    "Invalid Expense Ratios:",
    len(invalid_expense)
)

Invalid Expense Ratios: 0


In [30]:
print(
    performance_clean.isnull().sum()
)

amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64


In [31]:
performance_clean.to_csv(
    processed_path / "clean_performance.csv",
    index=False
)

print("clean_performance.csv saved")

clean_performance.csv saved
